Имеем приятный для первого раза датасет с информацией о аренде велосипедов в Сеуле за определенный период(интервал мы выясним самостоятельно), исследуем имеющиеся данные, посмотрим различные корреляции и посмотрим, можно ли построить предсказательную модель(регрессию) для кол-ва арендованных велосипедов, что могло бы оптимизировать работу прокатов, определить необходимое кол-во сотрудников(поддержка, кассиры, механики) в разное рабочее время.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("SeoulBikeData.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      8760 non-null   str    
 1   Rented Bike Count         8760 non-null   int64  
 2   Hour                      8760 non-null   int64  
 3   Temperature(C)            8760 non-null   float64
 4   Humidity(%)               8760 non-null   int64  
 5   Wind speed (m/s)          8760 non-null   float64
 6   Visibility (10m)          8760 non-null   int64  
 7   Dew point temperature(C)  8760 non-null   float64
 8   Solar Radiation (MJ/m2)   8760 non-null   float64
 9   Rainfall(mm)              8760 non-null   float64
 10  Snowfall (cm)             8760 non-null   float64
 11  Seasons                   8760 non-null   str    
 12  Holiday                   8760 non-null   str    
 13  Functioning Day           8760 non-null   str    
dtypes: float64(6), int6

Видим большое кол-во различных столбцов. Что приятно, датасет не имеет грязных данных, отсутствуют пропуски и опечатки(датасет тренировочный). Рассмотрим, что имеем: 

    Наш целевой столбец - Rented Bike Count(Кол-во арендованных за это время байков);
    - Date - Дата в формате dd.mm.yyyy(текст);
    - Hour - Час из расчета от 0 до 24, целочисленный;
    - Temperature(C) - Температура в градусах Цельсия в данный час;
    - Humidity(%) - Влажность воздуха;
    - Wind speed (m/s) - Скорость ветра;
    - Visibility (10m) - Видимость, однако в таблице не условная единица размером в десяток метров, как выяснилось, это высота установки прибора, 2000 здесь означает именно 2000 метров;
    - Dew point temperature(C) - Точка росы. Это та критическая температура, при которой воздух достигает 100% влажности (насыщения) и больше не может удерживать пар. Излишек выпадает в виде конденсата (росы, инея, тумана).
    - Solar Radiation (MJ/m2) - интенсивность солнечного излучения(MJ = мегаджоуль (единица энергии) / m² = на квадратный метр). Чем больше число, тем ярче солнце и жарче от него.
    - Rainfall(mm)/Snowfall (cm) - Кол-во осадков в виде дождя/снега. 
    - Seasons - время года(весна, осень, лето, зима)
    - Holiday - не путать, это государственные праздники, не суббота/воскресенье. 
    - Functioning day - рабочий/не рабочий день.

Возникает желание избавиться от лишних по моему мнению столбцов, таких как солнечное излучение или точка росы, но мы не можем утверждать, что определенная группа лиц на них не ориентируется. Вдруг это важно спортсменам, или людям для кожи которых имеет значение Solar Radiation? 

Разберем этот момент позже.

In [3]:
df.describe()

,Rented Bike Count,Hour,Temperature(C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm)
count,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000
mean,704.602055,11.500000,12.882922,58.226256,1.724909,1436.825799,4.073813,0.569111,0.148687,0.075068
std,644.997468,6.922582,11.944825,20.362413,1.036300,608.298712,13.060369,0.868746,1.128193,0.436746
min,0.000000,0.000000,-17.800000,0.000000,0.000000,27.000000,-30.600000,0.000000,0.000000,0.000000
25%,191.000000,5.750000,3.500000,42.000000,0.900000,940.000000,-4.700000,0.000000,0.000000,0.000000
50%,504.500000,11.500000,13.700000,57.000000,1.500000,1698.000000,5.100000,0.010000,0.000000,0.000000
75%,1065.250000,17.250000,22.500000,74.000000,2.300000,2000.000000,14.800000,0.930000,0.000000,0.000000
max,3556.000000,23.000000,39.400000,98.000000,7.400000,2000.000000,27.200000,3.520000,35.000000,8.800000


Убеждаемся, что в числовых данных у нас нет странных выбросов, аномальных значений на порогах. Теперь рассмотрим текстовые данные на предмет опечаток некоторого рода(например вместо Yes написано Yess).

In [4]:
df.describe(include='object')

/tmp/ipykernel_956171/87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,Date,Seasons,Holiday,Functioning Day
count,8760,8760,8760,8760
unique,365,4,2,2
top,01/12/2017,Spring,No Holiday,Yes
freq,24,2208,8328,8465


Можно сделать вывод, что данные в контексте грубых опечаток корректны, и таковых не имеют. Мы видим что по датам у нас столько уникальных значений, сколько дней в году, такую же картину наблюдаем и по остальным признакам. 